# Can a CNN still read obfuscated digits? Rerun with per-seed results

A small replication of McPherson, Shokri and Shmatikov (2016), *Defeating Image Obfuscation with Deep Learning*, on MNIST.

This notebook repeats the experiment of `dl_obfuscation.ipynb` with the same data, obfuscations, model, optimizer and batch size, and changes how the experiment is run and recorded:

- It uses ten seeds on a GPU (0 to 9) instead of three, and one on a CPU.
- Every model trains for 10 epochs, and both models are tested after epochs 2 and 10. The models at epoch 2 are exactly those a 2-epoch run would train, and testing draws no random numbers, so pausing to test does not change the rest of training. The comparison between 2 and 10 epochs is therefore paired within one run. This replaces `dl_obfuscation_10epochs.ipynb`, which trained separate models on three conditions.
- `results_per_seed.csv` has one row per condition, seed and epoch, with integer counts of correct test answers for both models.
- The GPU kernels are deterministic, and `run_info.json` records the software versions, the GPU and CPU, and hashes of the obfuscated images.
- The analysis in section 8 reads only `results_per_seed.csv`.

To start a new run in Google Colab, choose Runtime → Change runtime type → T4 GPU → Save. Start from a fresh session (open the notebook anew, or Runtime → Restart session), because the first cell must configure determinism before PyTorch starts the GPU; it stops with an error if the GPU was already started. Set `RUN_TRAINING = True` in the first cell, then choose Runtime → Run all. Training takes about 25 minutes on a T4. Section 7 then downloads `results_per_seed.csv` and `run_info.json`. Colab deletes its files when the runtime ends, so keep both. Finally, choose File → Download → Download .ipynb so that the saved notebook keeps its printed outputs.

To rerun only the analysis, leave `RUN_TRAINING = False`, put `results_per_seed.csv` next to the notebook (in Colab: Files panel, the folder icon on the left → Upload), then choose Runtime → Run all. Nothing is trained.

This code was written by an AI coding assistant (Claude, by Anthropic).

In [1]:
RUN_TRAINING = False  # True: train every model (about 25-30 min on a T4). False: only analyse RESULTS_CSV.
RESULTS_CSV = 'results_per_seed.csv'
RUN_INFO = 'run_info.json'

import os, sys
# Deterministic cuBLAS needs this set before PyTorch starts the GPU; older PyTorch versions raise an error without it.
if 'CUBLAS_WORKSPACE_CONFIG' not in os.environ and 'torch' in sys.modules and sys.modules['torch'].cuda.is_initialized():
    raise RuntimeError('The GPU started before CUBLAS_WORKSPACE_CONFIG was set: Runtime → Restart session, then run all.')
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import copy, csv, datetime, hashlib, json, platform, random, subprocess, time
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

torch.use_deterministic_algorithms(True)   # raise an error on any operation without a deterministic version
torch.backends.cudnn.benchmark = False      # choose convolution algorithms the same way every time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 10             # passes over the 60,000 training images
EVAL_EPOCHS = (2, 10)   # test both models after these epochs
BATCH = 128             # images per training step
LR = 1e-3               # Adam learning rate
SEEDS = list(range(10)) if device == 'cuda' else [0]   # 10 repeats on a GPU, 1 on a CPU
print('Device:', device, '| seeds:', SEEDS, '| epochs:', EPOCHS, '| tested after epochs:', EVAL_EPOCHS,
      '| RUN_TRAINING:', RUN_TRAINING)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

Device: cuda | seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | epochs: 10 | tested after epochs: (2, 10) | RUN_TRAINING: True


## 1. Environment

The environment is saved to `run_info.json` with the results. Runs on different GPU types can differ even with deterministic kernels: the random batch order can be generated differently, and on Ampere and newer GPUs (A100, L4) PyTorch runs convolutions in TF32 by default.

In [2]:
def cpu_model():
    try:
        with open('/proc/cpuinfo') as fh:
            for line in fh:
                if line.startswith('model name'):
                    return line.split(':', 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()

def gpu_driver():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=30)
        return out.stdout.strip() or None
    except (OSError, subprocess.SubprocessError):
        return None

gpu = torch.cuda.get_device_properties(0) if device == 'cuda' else None
ENV = {
    'recorded_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scipy': scipy.__version__,
    'device': device,
    'cpu': cpu_model(),
    'cpu_capability': torch.backends.cpu.get_cpu_capability(),   # instruction set PyTorch uses on the CPU
    'cpu_threads': torch.get_num_threads(),
    'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version() if gpu else None,
    'gpu': gpu.name if gpu else None,
    'gpu_sm_count': gpu.multi_processor_count if gpu else None,
    'gpu_compute_capability': f'{gpu.major}.{gpu.minor}' if gpu else None,
    'gpu_driver': gpu_driver() if gpu else None,
    'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
    'cudnn_benchmark': torch.backends.cudnn.benchmark,
    'tf32_matmul': torch.backends.cuda.matmul.allow_tf32,
    'tf32_cudnn': torch.backends.cudnn.allow_tf32,
    'cublas_workspace_config': os.environ.get('CUBLAS_WORKSPACE_CONFIG'),
    'epochs': EPOCHS,
    'tested_after_epochs': list(EVAL_EPOCHS),
    'batch': BATCH,
    'learning_rate': LR,
    'seeds': SEEDS,
}
print(json.dumps(ENV, indent=1))

{
 "recorded_utc": "2026-10-22T02:53:17+00:00",
 "python": "3.13.15",
 "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
 "torch": "2.11.0+cu130",
 "torchvision": "0.26.0+cu130",
 "numpy": "2.1.3",
 "pandas": "2.2.3",
 "scipy": "1.16.3",
 "device": "cuda",
 "cpu": "Intel(R) Xeon(R) CPU @ 2.00GHz",
 "cpu_capability": "AVX512",
 "cpu_threads": 1,
 "cuda": "13.0",
 "cudnn": 92700,
 "gpu": "Tesla T4",
 "gpu_sm_count": 40,
 "gpu_compute_capability": "7.5",
 "gpu_driver": "580.82.07",
 "deterministic_algorithms": true,
 "cudnn_benchmark": false,
 "tf32_matmul": false,
 "tf32_cudnn": true,
 "cublas_workspace_config": ":4096:8",
 "epochs": 10,
 "tested_after_epochs": [
  2,
  10
 ],
 "batch": 128,
 "learning_rate": 0.001,
 "seeds": [
  0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9
 ]
}


## 2. Data: MNIST (60,000 training and 10,000 test images of handwritten digits, 28×28 pixels)

In [3]:
train = torchvision.datasets.MNIST('data', train=True, download=True)
test = torchvision.datasets.MNIST('data', train=False, download=True)
Xtr = train.data.float().div(255).unsqueeze(1)   # 60000 x 1 x 28 x 28, pixel values 0..1
ytr = train.targets
Xte = test.data.float().div(255).unsqueeze(1)    # 10000 x 1 x 28 x 28
yte = test.targets
print('Train:', tuple(Xtr.shape), '| Test:', tuple(Xte.shape))

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 488kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.50MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.82MB/s]


Train: (60000, 1, 28, 28) | Test: (10000, 1, 28, 28)


## 3. The obfuscations

The obfuscations are unchanged from `dl_obfuscation.ipynb`. They run on the CPU, even in a GPU run. `run_info.json` records a hash of every obfuscated image set, because the last bits of the blur can depend on the CPU's instruction set.

In [4]:
def pixelate(x, block):
    """Mosaic: average each block x block square, then scale it back up to 28 x 28."""
    return F.interpolate(F.avg_pool2d(x, block), scale_factor=block, mode='nearest')

def blur(x, sigma):
    """Gaussian blur with standard deviation sigma, in pixels."""
    radius = int(3 * sigma)
    t = torch.arange(-radius, radius + 1, dtype=torch.float32)
    g = torch.exp(-t ** 2 / (2 * sigma ** 2))
    g = g / g.sum()
    k = len(g)
    x = F.pad(x, (radius, radius, radius, radius), mode='reflect')
    x = F.conv2d(x, g.view(1, 1, 1, k))   # blur along each row
    x = F.conv2d(x, g.view(1, 1, k, 1))   # then along each column
    return x

def solid_mask(x):
    """Cover the whole digit with a solid black box: no pixel of it survives."""
    return torch.zeros_like(x)

CONDITIONS = {                     # 'clean' comes first: its models are also the attacker trained on clean images
    'clean': lambda x: x,
    'pixelate 2x2': lambda x: pixelate(x, 2),
    'pixelate 4x4': lambda x: pixelate(x, 4),
    'pixelate 7x7': lambda x: pixelate(x, 7),
    'pixelate 14x14': lambda x: pixelate(x, 14),
    'blur sigma=1': lambda x: blur(x, 1),
    'blur sigma=2': lambda x: blur(x, 2),
    'blur sigma=4': lambda x: blur(x, 4),
    'solid mask': solid_mask,
}

## 4. The model: a small CNN (two convolution + pooling layers, then two fully connected layers)

The model and each training step are unchanged. Training now pauses after the epochs in `EVAL_EPOCHS` so both models can be tested, and it hashes the initial weights and the batch order, so the analysis can check that every condition within a seed shared them.

In [5]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)    # 32 filters, 3x3
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)   # 64 filters, 3x3
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)                  # one output per digit

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)     # 28x28 -> 14x14
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)     # 14x14 -> 7x7
        x = self.drop(F.relu(self.fc1(x.flatten(1))))
        return self.fc2(x)

def sha(*tensors):
    """First 16 hex digits of the SHA-256 of the tensors' bytes."""
    h = hashlib.sha256()
    for t in tensors:
        h.update(t.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()[:16]

def train_model(X, y, seed):
    """Train one model for EPOCHS epochs. After each epoch in EVAL_EPOCHS, pause and yield
    (epoch, model, hash of the initial weights, hash of the batch order so far). Testing the
    model during a pause draws no random numbers, so it does not change the rest of training."""
    set_seed(seed)
    model = SmallCNN().to(device)
    init_sha = sha(*model.state_dict().values())
    order = hashlib.sha256()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(X), device=X.device)
        order.update(perm.cpu().numpy().tobytes())
        for i in range(0, len(X), BATCH):
            idx = perm[i:i + BATCH]
            opt.zero_grad()
            loss = F.cross_entropy(model(X[idx]), y[idx])
            loss.backward()
            opt.step()
        if epoch in EVAL_EPOCHS:
            yield epoch, model, init_sha, order.hexdigest()[:16]

@torch.no_grad()
def evaluate(model, X, y):
    """Number of correct answers, and the digit the model gave for every image if it gave only one (else None)."""
    model.eval()
    pred = torch.cat([model(X[i:i + 1000]).argmax(1) for i in range(0, len(X), 1000)])
    answer = int(pred[0]) if bool((pred == pred[0]).all()) else None
    return int((pred == y).sum()), answer

## 5. Pre-flight check (only when `RUN_TRAINING` is True)

This cell runs one training step, one evaluation and every obfuscation under the deterministic settings, so an operation without a deterministic version stops the run here rather than partway through. It also refuses to overwrite the results of an earlier run.

In [6]:
if RUN_TRAINING:
    assert next(iter(CONDITIONS)) == 'clean', "'clean' must come first: its models are the attacker trained on clean images"
    for path in (RESULTS_CSV, RUN_INFO):
        if os.path.exists(path):
            raise FileExistsError(f'{path} already exists: rename or move it before a new run')
    for f in CONDITIONS.values():
        f(Xtr[:4])
    model = SmallCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    model.train()
    opt.zero_grad()
    F.cross_entropy(model(Xtr[:BATCH].to(device)), ytr[:BATCH].to(device)).backward()
    opt.step()
    torch.randperm(len(Xtr), device=device)
    evaluate(copy.deepcopy(model), Xte[:1000].to(device), yte[:1000].to(device))
    print('Pre-flight check passed on', device)
else:
    print('RUN_TRAINING is False: no pre-flight check.')

Pre-flight check passed on cuda


## 6. The experiment (only when `RUN_TRAINING` is True)

For each obfuscation, two models are tested on obfuscated test images: one trained on clean images (a naive attacker that has never seen obfuscation) and one trained on obfuscated images with their labels (the attacker from McPherson et al.).

Within a seed, every model starts from the same initial weights and sees the training batches in the same order, so the conditions differ only in the images. Each row of `results_per_seed.csv` is one condition, seed and epoch:

| Column | Meaning |
|---|---|
| `condition`, `seed`, `epoch` | The obfuscation, the seed, and the epoch after which both models were tested |
| `n_test` | Number of test images (10,000) |
| `correct_trained_on_clean` | Correct answers of the model trained on clean images, tested on this condition |
| `correct_trained_on_obfuscated` | Correct answers of the model trained on this condition |
| `answer_trained_on_clean`, `answer_trained_on_obfuscated` | The digit the model gave for every test image, if it gave only one; empty if its answers varied |
| `init_sha`, `order_sha` | Hashes of the initial weights and of the batch order up to this epoch |

The file holds no timings, so two runs on the same hardware and software give byte-identical files. `run_info.json` holds the training time.

In [7]:
def mean_accuracy(rows, condition, epoch, column):
    return np.mean([100 * r[column] / r['n_test'] for r in rows if r['condition'] == condition and r['epoch'] == epoch])

if RUN_TRAINING:
    ytr_d, yte_d = ytr.to(device), yte.to(device)
    rows, clean_models, data_sha = [], {}, {}
    print('Mean test accuracy over seeds: trained on clean / trained on obfuscated')
    t_start = time.time()
    for name, f in CONDITIONS.items():
        t0 = time.time()
        Xtr_o, Xte_o = f(Xtr), f(Xte)
        data_sha[name] = {'train': sha(Xtr_o), 'test': sha(Xte_o)}
        Xtr_o, Xte_o = Xtr_o.to(device), Xte_o.to(device)
        for s in SEEDS:
            for epoch, model, init_sha, order_sha in train_model(Xtr_o, ytr_d, s):
                if name == 'clean':
                    clean_models[s, epoch] = copy.deepcopy(model)
                correct_clean, answer_clean = evaluate(clean_models[s, epoch], Xte_o, yte_d)
                correct_obfuscated, answer_obfuscated = evaluate(model, Xte_o, yte_d)
                rows.append({'condition': name, 'seed': s, 'epoch': epoch, 'n_test': len(yte_d),
                             'correct_trained_on_clean': correct_clean,
                             'correct_trained_on_obfuscated': correct_obfuscated,
                             'answer_trained_on_clean': answer_clean,
                             'answer_trained_on_obfuscated': answer_obfuscated,
                             'init_sha': init_sha, 'order_sha': order_sha})
        means = '   '.join(f'epoch {e}: {mean_accuracy(rows, name, e, "correct_trained_on_clean"):6.2f}% / '
                           f'{mean_accuracy(rows, name, e, "correct_trained_on_obfuscated"):6.2f}%' for e in EVAL_EPOCHS)
        print(f'{name:15s} {means}   ({time.time() - t0:.0f} s)')
    minutes = (time.time() - t_start) / 60
    print(f'Total: {minutes:.1f} min')

    with open(RESULTS_CSV, 'w', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]), lineterminator='\n')
        writer.writeheader()
        writer.writerows(rows)
    with open(RESULTS_CSV, 'rb') as fh:
        csv_sha = hashlib.sha256(fh.read()).hexdigest()
    with open(RUN_INFO, 'w') as fh:
        json.dump({**ENV, 'training_minutes': round(minutes, 1), 'results_csv_sha256': csv_sha,
                   'obfuscated_data_sha256': data_sha}, fh, indent=1)
        fh.write('\n')
    print(f'Wrote {RESULTS_CSV} ({len(rows)} rows, SHA-256 {csv_sha[:16]}...) and {RUN_INFO}')
else:
    print('RUN_TRAINING is False: nothing trained.')

Mean test accuracy over seeds: trained on clean / trained on obfuscated
clean           epoch 2:  98.62% /  98.62%   epoch 10:  99.19% /  99.19%   (157 s)
pixelate 2x2    epoch 2:  98.12% /  98.39%   epoch 10:  98.77% /  99.04%   (158 s)
pixelate 4x4    epoch 2:  82.24% /  95.96%   epoch 10:  79.30% /  97.35%   (159 s)
pixelate 7x7    epoch 2:  35.34% /  83.98%   epoch 10:  33.79% /  87.59%   (159 s)
pixelate 14x14  epoch 2:  13.36% /  51.15%   epoch 10:  13.49% /  51.91%   (158 s)
blur sigma=1    epoch 2:  97.97% /  98.19%   epoch 10:  98.50% /  99.01%   (167 s)
blur sigma=2    epoch 2:  89.29% /  96.34%   epoch 10:  86.22% /  98.28%   (166 s)
blur sigma=4    epoch 2:  36.39% /  89.28%   epoch 10:  31.11% /  94.86%   (165 s)
solid mask      epoch 2:  10.79% /  11.35%   epoch 10:  10.78% /  11.35%   (152 s)
Total: 24.0 min
Wrote results_per_seed.csv (180 rows, SHA-256 e27e65acf259314b...) and run_info.json


## 7. Download the results (Colab only)

Colab deletes its files when the runtime ends. Commit both files with this notebook.

In [8]:
if RUN_TRAINING:
    try:
        from google.colab import files
        for path in (RESULTS_CSV, RUN_INFO):
            files.download(path)
    except ImportError:
        print("Not running on Colab: the files are in this notebook's folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Analysis (reads only `results_per_seed.csv`)

Accuracy is 100 × correct answers / `n_test`. The tables give the mean over seeds ± the sample SD, which divides by n − 1, and a 95% t-interval for the mean in brackets. The earlier `results.csv` and `results_10epochs.csv` gave population SDs, which divide by n and are smaller by a factor of √((n − 1)/n): 0.82 for their 3 seeds, 0.95 for 10. An interval that would pass 0% or 100% is cut there and marked \*. With one seed there is no SD or interval, and the tables show n = 1.

Gains and contrasts are paired: each difference is taken within a seed, between models that started from the same weights and saw the same batch order (checked below), and the mean, SD and interval are those of the per-seed differences.

In [9]:
ANSWERS = ['answer_trained_on_clean', 'answer_trained_on_obfuscated']
if not os.path.exists(RESULTS_CSV):
    raise FileNotFoundError(f'{RESULTS_CSV} not found: put it next to this notebook (Colab: Files panel → Upload), '
                            'or set RUN_TRAINING = True for a new run.')
res = pd.read_csv(RESULTS_CSV, dtype={c: 'Int64' for c in ANSWERS})
res['acc_clean'] = 100 * res['correct_trained_on_clean'] / res['n_test']
res['acc_obfuscated'] = 100 * res['correct_trained_on_obfuscated'] / res['n_test']
res['gain'] = res['acc_obfuscated'] - res['acc_clean']
conditions = list(dict.fromkeys(res['condition']))   # in the order they were run
epochs = sorted(res['epoch'].unique().tolist())
print(f'{RESULTS_CSV}: {len(res)} rows, {len(conditions)} conditions, '
      f'seeds {sorted(res["seed"].unique().tolist())}, tested after epochs {epochs}')

shared = res.groupby(['seed', 'epoch'])[['init_sha', 'order_sha']].nunique()
assert (shared == 1).all().all(), 'within a seed, the models did not all share their initial weights and batch order'
print('Within each seed, every model started from the same weights and saw the same batch order.')

def describe(x, bounds=(-100, 100)):
    """Mean, sample SD and 95% t-interval of x, the interval cut at bounds. The SD and interval are NaN for n = 1."""
    x = np.asarray(x, dtype=float)
    d = {'n': len(x), 'mean': x.mean(), 'sd': np.nan, 'low': np.nan, 'high': np.nan, 'cut': False}
    if len(x) > 1:
        d['sd'] = x.std(ddof=1)
        half = stats.t.ppf(0.975, len(x) - 1) * d['sd'] / np.sqrt(len(x))
        d['low'], d['high'] = max(d['mean'] - half, bounds[0]), min(d['mean'] + half, bounds[1])
        d['cut'] = bool(d['low'] > d['mean'] - half or d['high'] < d['mean'] + half)
    return d

def fmt(d):
    if d['n'] < 2:
        return f"{d['mean']:.2f} (n = {d['n']})"
    return f"{d['mean']:.2f} ± {d['sd']:.2f} [{d['low']:.2f}, {d['high']:.2f}]" + ('*' if d['cut'] else '')

def per_seed(column, condition, epoch):
    """Values of one column for one condition and epoch, indexed by seed."""
    return res[(res['condition'] == condition) & (res['epoch'] == epoch)].set_index('seed')[column]

results_per_seed.csv: 180 rows, 9 conditions, seeds [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], tested after epochs [2, 10]
Within each seed, every model started from the same weights and saw the same batch order.


### 8.1 Accuracy and paired gain under each condition

In [10]:
QUANTITIES = [('trained on clean', 'acc_clean', (0, 100)),
              ('trained on obfuscated', 'acc_obfuscated', (0, 100)),
              ('gain', 'gain', (-100, 100))]
summary = pd.DataFrame([{'epoch': e, 'condition': c, 'quantity': q, **describe(per_seed(col, c, e), bounds)}
                        for e in epochs for c in conditions for q, col, bounds in QUANTITIES])
for e in epochs:
    s = summary[summary['epoch'] == e]
    table = s.assign(text=s.apply(fmt, axis=1)).pivot(index='condition', columns='quantity', values='text')
    print(f'After {e} epochs: test accuracy (%), mean ± SD [95% interval]; gain = obfuscated minus clean, paired within seed')
    print(table.loc[conditions, [q for q, _, _ in QUANTITIES]].to_string(), end='\n\n')
if summary['cut'].any():
    print('* interval cut at 0% or 100%')

After 2 epochs: test accuracy (%), mean ± SD [95% interval]; gain = obfuscated minus clean, paired within seed
quantity                    trained on clean        trained on obfuscated                          gain
condition                                                                                              
clean            98.62 ± 0.09 [98.55, 98.68]  98.62 ± 0.09 [98.55, 98.68]      0.00 ± 0.00 [0.00, 0.00]
pixelate 2x2     98.12 ± 0.26 [97.94, 98.31]  98.39 ± 0.14 [98.28, 98.49]      0.26 ± 0.18 [0.14, 0.39]
pixelate 4x4     82.24 ± 4.21 [79.23, 85.25]  95.96 ± 0.12 [95.87, 96.05]   13.72 ± 4.23 [10.69, 16.75]
pixelate 7x7     35.34 ± 3.79 [32.63, 38.05]  83.98 ± 0.64 [83.52, 84.43]   48.64 ± 3.68 [46.00, 51.27]
pixelate 14x14   13.36 ± 3.07 [11.16, 15.56]  51.15 ± 0.40 [50.87, 51.44]   37.80 ± 3.09 [35.58, 40.01]
blur sigma=1     97.97 ± 0.23 [97.81, 98.14]  98.19 ± 0.11 [98.11, 98.27]      0.21 ± 0.20 [0.07, 0.36]
blur sigma=2     89.29 ± 3.89 [86.51, 92.07]  96.34 ± 0.2

### 8.2 Blur σ = 4 against pixelate 7×7

These are within-seed differences, σ = 4 minus 7×7, in percentage points.

In [11]:
A, B = 'blur sigma=4', 'pixelate 7x7'
if {A, B} <= set(conditions):
    contrast = {f'after {e} epochs, {q}': fmt(describe((per_seed(col, A, e) - per_seed(col, B, e)).dropna()))
                for e in epochs for q, col, _ in QUANTITIES}
    print(f'{A} minus {B}, paired within seed: mean ± SD [95% interval]')
    print(pd.Series(contrast).to_string())
else:
    print(f'{A} or {B} is not in {RESULTS_CSV}.')

blur sigma=4 minus pixelate 7x7, paired within seed: mean ± SD [95% interval]
after 2 epochs, trained on clean           1.06 ± 8.76 [-5.21, 7.32]
after 2 epochs, trained on obfuscated       5.31 ± 0.89 [4.67, 5.94]
after 2 epochs, gain                      4.25 ± 9.05 [-2.22, 10.73]
after 10 epochs, trained on clean         -2.68 ± 6.48 [-7.31, 1.96]
after 10 epochs, trained on obfuscated      7.27 ± 0.33 [7.03, 7.51]
after 10 epochs, gain                      9.94 ± 6.59 [5.23, 14.66]


### 8.3 Epoch 2 against epoch 10

This gives the change in accuracy from the first to the last tested epoch, paired within seed. The last line answers the question of the earlier 10-epoch check: does more training help the model trained on obfuscated images more at blur σ = 4 than at pixelate 7×7?

In [12]:
if len(epochs) < 2:
    print(f'{RESULTS_CSV} has one tested epoch, so there is nothing to compare.')
else:
    e0, e1 = epochs[0], epochs[-1]

    def change(column, condition):
        return (per_seed(column, condition, e1) - per_seed(column, condition, e0)).dropna()

    table = pd.DataFrame({c: {'trained on clean': fmt(describe(change('acc_clean', c))),
                              'trained on obfuscated': fmt(describe(change('acc_obfuscated', c)))}
                          for c in conditions}).T
    print(f'Change from epoch {e0} to epoch {e1}, paired within seed (percentage points): mean ± SD [95% interval]')
    print(table.to_string())
    if {A, B} <= set(conditions):
        extra = (change('acc_obfuscated', A) - change('acc_obfuscated', B)).dropna()
        print(f'\nTrained on obfuscated images, (change at {A}) minus (change at {B}): {fmt(describe(extra))}')

Change from epoch 2 to epoch 10, paired within seed (percentage points): mean ± SD [95% interval]
                            trained on clean     trained on obfuscated
clean               0.57 ± 0.07 [0.52, 0.62]  0.57 ± 0.07 [0.52, 0.62]
pixelate 2x2        0.65 ± 0.21 [0.50, 0.80]  0.65 ± 0.15 [0.54, 0.76]
pixelate 4x4      -2.94 ± 4.46 [-6.12, 0.25]  1.39 ± 0.19 [1.25, 1.52]
pixelate 7x7      -1.55 ± 4.26 [-4.60, 1.50]  3.62 ± 0.64 [3.16, 4.08]
pixelate 14x14     0.13 ± 4.25 [-2.91, 3.17]  0.76 ± 0.54 [0.38, 1.14]
blur sigma=1        0.53 ± 0.42 [0.23, 0.82]  0.82 ± 0.13 [0.73, 0.91]
blur sigma=2      -3.07 ± 6.16 [-7.47, 1.34]  1.94 ± 0.27 [1.74, 2.13]
blur sigma=4    -5.28 ± 10.08 [-12.49, 1.93]  5.58 ± 0.57 [5.17, 5.99]
solid mask        -0.01 ± 1.02 [-0.74, 0.72]  0.00 ± 0.00 [0.00, 0.00]

Trained on obfuscated images, (change at blur sigma=4) minus (change at pixelate 7x7): 1.96 ± 0.95 [1.28, 2.64]


### 8.4 Constant answers

Every masked image is the same, so each model gives one digit for all of them. This cell lists the digit each model gave, by seed and epoch, and any other model that gave a single answer.

In [13]:
mask = res[res['condition'] == 'solid mask']
if mask.empty:
    print(f'No solid-mask rows in {RESULTS_CSV}.')
else:
    print('Solid mask: the digit each model gave for every test image')
    print(mask.pivot(index='seed', columns='epoch', values=ANSWERS).to_string())
    for e in epochs:
        for col in ANSWERS:
            counts = mask.loc[mask['epoch'] == e, col].value_counts().sort_index()
            print(f'after {e} epochs, {col[len("answer_"):].replace("_", " ")}: '
                  + ', '.join(f'"{k}" in {v} seed{"s" if v != 1 else ""}' for k, v in counts.items()))
others = res[(res['condition'] != 'solid mask') & res[ANSWERS].notna().any(axis=1)]
print('\nOther models that gave a single answer:',
      'none' if others.empty else '\n' + others[['condition', 'seed', 'epoch'] + ANSWERS].to_string(index=False))

Solid mask: the digit each model gave for every test image
      answer_trained_on_clean    answer_trained_on_obfuscated   
epoch                      2  10                           2  10
seed                                                            
0                           1  1                            1  1
1                           8  0                            1  1
2                           1  0                            1  1
3                           0  0                            1  1
4                           5  1                            1  1
5                           1  2                            1  1
6                           1  1                            1  1
7                           1  1                            1  1
8                           1  1                            1  1
9                           1  1                            1  1
after 2 epochs, trained on clean: "0" in 1 seed, "1" in 7 seeds, "5" in 1 seed, "8" in 1 seed
af